# Demo

In [1]:
# Initial Demo
"""
PRENATALPPKT ETL PIPELINE
Observer JSON → TermBins → Phenopacket v2.0

Uses the official GA4GH phenopackets library per:
https://phenopacket-schema.readthedocs.io/en/latest/python.html
"""

import json
import re
from datetime import datetime, timezone
from pathlib import Path

from google.protobuf.json_format import MessageToJson
from google.protobuf.timestamp_pb2 import Timestamp
import phenopackets.schema.v2 as pps2

from prenatalppkt.etl.extractors import observer
from prenatalppkt.gestational_age import GestationalAge

print("=" * 80)
print("PRENATALPPKT ETL PIPELINE")
print("Observer JSON → TermBins → Phenopacket v2.0")
print("=" * 80)

# -----------------------------------------------------------------------------
# STEP 1: Load Apple Sally Observer JSON
# -----------------------------------------------------------------------------
print("\n STEP 1: Loading Observer JSON...")

data_path = Path("tests/data/Apple_Sally_pretty.json")
with open(data_path) as f:
    observer_data = json.load(f)

print(f"Loaded: {data_path}")
print(f"Fetuses: {len(observer_data.get('fetuses', []))}")

first_fetus = observer_data["fetuses"][0]
measurements = first_fetus.get("measurements", [])
print(f"Measurements: {len(measurements)}")
print(
    f"Sample: {measurements[0]['label']} = "
    f"{measurements[0]['value']} {measurements[0]['unit_of_measure']}"
)

# -----------------------------------------------------------------------------
# STEP 2: Extract TermBins using Observer extractor
# -----------------------------------------------------------------------------
print("\n  STEP 2: Extracting biometry measurements to TermBins...")

term_bins = observer.extract(observer_data)
print(f" Extracted {len(term_bins)} TermBins")

for i, tb in enumerate(term_bins, 1):
    print(f"\n  [{i}] {tb.description}")
    print(f"      HPO: {tb.hpo_id} - {tb.hpo_label}")
    print(f"      Normal: {tb.normal}")

# -----------------------------------------------------------------------------
# STEP 3: Convert TermBins → Phenotypic Features (using phenopackets library)
# -----------------------------------------------------------------------------
print("\n STEP 3: Converting TermBins to PhenotypicFeatures...")


def parse_ga_from_description(description: str) -> tuple[int, int]:
    """Extract weeks and days from TermBin description."""
    match = re.search(r"at (\d+)w(\d+)d", description)
    if match:
        return int(match.group(1)), int(match.group(2))
    # Fallback
    first_m = observer_data["fetuses"][0]["measurements"][0]
    ga = GestationalAge.from_weeks(first_m.get("calculated_ega", 26.9))
    return ga.weeks, ga.days


phenotypic_features = []

for tb in term_bins:
    weeks, days = parse_ga_from_description(tb.description)

    # Create GestationalAge message
    gestational_age = pps2.GestationalAge(weeks=weeks, days=days)

    # Create TimeElement with gestational_age
    onset = pps2.TimeElement(gestational_age=gestational_age)

    # Create OntologyClass for the HPO term
    hpo_type = pps2.OntologyClass(id=tb.hpo_id, label=tb.hpo_label)

    # Create PhenotypicFeature
    pf = pps2.PhenotypicFeature(
        type=hpo_type,
        excluded=tb.normal,  # If normal=True, abnormality is excluded
        onset=onset,
        description=tb.description,
    )

    phenotypic_features.append(pf)

print(f" Generated {len(phenotypic_features)} PhenotypicFeatures")

for i, pf in enumerate(phenotypic_features, 1):
    status = "EXCLUDED (normal)" if pf.excluded else "OBSERVED (abnormal)"
    print(f"\n  [{i}] {pf.type.id}")
    print(f"      Status: {status}")
    print(f"      Description: {pf.description}")

# -----------------------------------------------------------------------------
# STEP 4: Build Complete Phenopacket v2.0
# -----------------------------------------------------------------------------
print("\n STEP 4: Building Phenopacket v2.0...")

# Get subject GA from first measurement
first_measurement = observer_data["fetuses"][0]["measurements"][0]
subject_ga_weeks = first_measurement.get("calculated_ega", 26.9)
subject_ga = GestationalAge.from_weeks(subject_ga_weeks)

# Create Individual (subject) with GestationalAge
subject_time = pps2.TimeElement(
    gestational_age=pps2.GestationalAge(weeks=subject_ga.weeks, days=subject_ga.days)
)

subject = pps2.Individual(
    id="fetus-1",
    sex=pps2.Sex.UNKNOWN_SEX,
    time_at_last_encounter=subject_time,
)

# Create timestamp for metadata
now = datetime.now(timezone.utc)
created_timestamp = Timestamp()
created_timestamp.FromDatetime(now)

# Create HPO Resource
hpo_resource = pps2.Resource(
    id="hp",
    name="Human Phenotype Ontology",
    url="http://purl.obolibrary.org/obo/hp.owl",
    version="2025-11-24",
    namespace_prefix="HP",
    iri_prefix="http://purl.obolibrary.org/obo/HP_",
)

# Create MetaData
metadata = pps2.MetaData(
    created=created_timestamp,
    created_by="prenatalppkt-etl-pipeline",
    phenopacket_schema_version="2.0",
)
metadata.resources.append(hpo_resource)

# Create the Phenopacket
phenopacket = pps2.Phenopacket(
    id="apple-sally-fetus-1",
    subject=subject,
    meta_data=metadata,
)
phenopacket.phenotypic_features.extend(phenotypic_features)

print("✓ Phenopacket created successfully")

# -----------------------------------------------------------------------------
# STEP 5: Display Results as JSON
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print(" PHENOPACKET v2.0 OUTPUT (JSON)")
print("=" * 80)

# Convert protobuf message to JSON using official method
phenopacket_json = MessageToJson(phenopacket, preserving_proto_field_name=True)
print(phenopacket_json)

# -----------------------------------------------------------------------------
# STEP 6: Validation Summary
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print(" VALIDATION SUMMARY")
print("=" * 80)

print("\n Phenopacket Structure:")
print(f"   ID: {phenopacket.id}")
print(f"   Subject ID: {phenopacket.subject.id}")
print(f"   Subject GA: {subject_ga.weeks}w{subject_ga.days}d")
print(f"   Sex: {pps2.Sex.Name(phenopacket.subject.sex)}")
print(f"   Phenotypic Features: {len(phenopacket.phenotypic_features)}")
print(f"   Schema Version: {phenopacket.meta_data.phenopacket_schema_version}")
print(f"   HPO Resource: {phenopacket.meta_data.resources[0].version}")

print("\n Phenotypic Features Detail:")
for i, pf in enumerate(phenopacket.phenotypic_features, 1):
    status = " Normal (excluded)" if pf.excluded else "Abnormal (observed)"
    ga = pf.onset.gestational_age
    print(f"\n  [{i}] {pf.type.id} - {pf.type.label}")
    print(f"      {status}")
    print(f"      Onset: {ga.weeks}w{ga.days}d")
    print(f"      Detail: {pf.description}")

# Count normal vs abnormal
normal_count = sum(1 for pf in phenopacket.phenotypic_features if pf.excluded)
abnormal_count = len(phenopacket.phenotypic_features) - normal_count

print("\n Summary Statistics:")
print(f"  Total features: {len(phenopacket.phenotypic_features)}")
print(f"  Normal (excluded): {normal_count}")
print(f"  Abnormal (observed): {abnormal_count}")

print("\n" + "=" * 80)
print(" SUCCESS: Valid Phenopacket v2.0 generated")
print("=" * 80)

# Save to file
output_path = Path("output/apple_sally_phenopacket_v2.json")
output_path.parent.mkdir(exist_ok=True)
with open(output_path, "w") as f:
    f.write(phenopacket_json)
print(f"\n Phenopacket saved to: {output_path}")

# Validate by round-tripping
print("\n Validation: Round-trip test...")
from google.protobuf.json_format import Parse

parsed_back = Parse(phenopacket_json, pps2.Phenopacket())
assert parsed_back.id == phenopacket.id
assert len(parsed_back.phenotypic_features) == len(phenopacket.phenotypic_features)
print(" Validation passed")

DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for head_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for biparietal_diameter
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for femur_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for abdominal_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for occipitofrontal_diameter
DEBUG:prenatalppkt.etl.term_bin_factory:Loaded mappings for: ['head_circumference', 'biparietal_diameter', 'femur_length', 'abdominal_circumference', 'occipitofrontal_diameter']
DEBUG:prenatalppkt.etl.extractors.observer:Starting Observer JSON extraction
DEBUG:prenatalppkt.etl.extractors.observer:Processing fetus 1
DEBUG:prenatalppkt.etl.extractors.observer:Found 6 measurements
DEBUG:prenatalppkt.etl.extractors.observer:Processing measurement: AC
DEBUG:prenatalppkt.etl.extractors.observer:AC has percentile=55.6% (valid)
DEBUG:prenatalppkt.etl.extractors.observer:Creating TermBin for AC: value=226.20000000000002mm, percentile=55.6%,

PRENATALPPKT ETL PIPELINE
Observer JSON → TermBins → Phenopacket v2.0

 STEP 1: Loading Observer JSON...
Loaded: tests/data/Apple_Sally_pretty.json
Fetuses: 1
Measurements: 6
Sample: AC = 22.62 cm

  STEP 2: Extracting biometry measurements to TermBins...
 Extracted 4 TermBins

  [1] AC: 226.2 mm (55.6%) at 26w6d [Fetus 1]
      HPO: HP:0034207 - Abnormal fetal gastrointestinal system morphology
      Normal: True

  [2] BPD: 66.8 mm (51.2%) at 26w6d [Fetus 1]
      HPO: HP:0000240 - Abnormality of skull size
      Normal: True

  [3] HC: 250.0 mm (42.5%) at 26w6d [Fetus 1]
      HPO: HP:0000240 - Abnormality of skull size
      Normal: True

  [4] Femur: 50.1 mm (46.8%) at 27w0d [Fetus 1]
      HPO: HP:0002823 - Abnormal femur morphology
      Normal: True

 STEP 3: Converting TermBins to PhenotypicFeatures...
 Generated 4 PhenotypicFeatures

  [1] HP:0034207
      Status: EXCLUDED (normal)
      Description: AC: 226.2 mm (55.6%) at 26w6d [Fetus 1]

  [2] HP:0000240
      Status: EXCL

In [2]:
# Shorter Test

import json
import re
from datetime import datetime, timezone
from pathlib import Path
from google.protobuf.json_format import MessageToJson
from google.protobuf.timestamp_pb2 import Timestamp
import phenopackets.schema.v2 as pps2
from prenatalppkt.etl.extractors import observer
from prenatalppkt.gestational_age import GestationalAge

print("\n STEP 1: Loading Observer JSON...")
data_path = Path("tests/data/Apple_Sally_pretty.json")
with open(data_path) as f:
    observer_data = json.load(f)
print(f"Loaded: {data_path}")
print(f"Fetuses: {len(observer_data.get('fetuses', []))}")

first_fetus = observer_data["fetuses"][0]
measurements = first_fetus.get("measurements", [])
print(f"Measurements: {len(measurements)}")
print(f"Sample: {measurements[0]['label']} = ", f"{measurements[0]['value']} {measurements[0]['unit_of_measure']}")

print("\n  STEP 2: Extracting biometry measurements to TermBins...")
term_bins = observer.extract(observer_data)
print(f" Extracted {len(term_bins)} TermBins")
for i, tb in enumerate(term_bins, 1):
    print(f"\n  [{i}] {tb.description}")
    print(f"      HPO: {tb.hpo_id} - {tb.hpo_label}")
    print(f"      Normal: {tb.normal}")

print("\n STEP 3: Converting TermBins to PhenotypicFeatures...")
def parse_ga_from_description(description: str) -> tuple[int, int]:
    """Extract weeks and days from TermBin description."""
    match = re.search(r"at (\d+)w(\d+)d", description)
    if match:
        return int(match.group(1)), int(match.group(2))
    # Fallback
    first_m = observer_data["fetuses"][0]["measurements"][0]
    ga = GestationalAge.from_weeks(first_m.get("calculated_ega", 26.9))
    return ga.weeks, ga.days
phenotypic_features = []
for tb in term_bins:
    weeks, days = parse_ga_from_description(tb.description)
    # Create GestationalAge message
    gestational_age = pps2.GestationalAge(weeks=weeks, days=days)
    # Create TimeElement with gestational_age
    onset = pps2.TimeElement(gestational_age=gestational_age)
    # Create OntologyClass for the HPO term
    hpo_type = pps2.OntologyClass(id=tb.hpo_id, label=tb.hpo_label)
    # Create PhenotypicFeature
    pf = pps2.PhenotypicFeature( type=hpo_type, excluded=tb.normal, onset=onset, description=tb.description)
    phenotypic_features.append(pf)
print(f" Generated {len(phenotypic_features)} PhenotypicFeatures")
for i, pf in enumerate(phenotypic_features, 1):
    status = "EXCLUDED (normal)" if pf.excluded else "OBSERVED (abnormal)"
    print(f"\n  [{i}] {pf.type.id}")
    print(f"      Status: {status}")
    print(f"      Description: {pf.description}")


print("\n STEP 4: Building Phenopacket v2.0...")
# Get subject GA from first measurement
first_measurement = observer_data["fetuses"][0]["measurements"][0]
subject_ga_weeks = first_measurement.get("calculated_ega", 26.9)
subject_ga = GestationalAge.from_weeks(subject_ga_weeks)
# Create Individual (subject) with GestationalAge
subject_time = pps2.TimeElement(gestational_age=pps2.GestationalAge(weeks=subject_ga.weeks, days=subject_ga.days))

subject = pps2.Individual(id="fetus-1", sex=pps2.Sex.UNKNOWN_SEX, time_at_last_encounter=subject_time)

# Create timestamp for metadata
now = datetime.now(timezone.utc)
created_timestamp = Timestamp()
created_timestamp.FromDatetime(now)

# Create HPO Resource
hpo_resource = pps2.Resource(id="hp", name="Human Phenotype Ontology", url="http://purl.obolibrary.org/obo/hp.owl", version="2025-11-24", namespace_prefix="HP", iri_prefix="http://purl.obolibrary.org/obo/HP_")

# Create MetaData
metadata = pps2.MetaData(created=created_timestamp, created_by="prenatalppkt-etl-pipeline", phenopacket_schema_version="2.0")
metadata.resources.append(hpo_resource)

# Create the Phenopacket
phenopacket = pps2.Phenopacket(id="apple-sally-fetus-1", subject=subject, meta_data=metadata)
phenopacket.phenotypic_features.extend(phenotypic_features)

# Convert protobuf message to JSON using official method
phenopacket_json = MessageToJson(phenopacket, preserving_proto_field_name=True)
print(phenopacket_json)

DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for head_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for biparietal_diameter
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for femur_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for abdominal_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for occipitofrontal_diameter
DEBUG:prenatalppkt.etl.term_bin_factory:Loaded mappings for: ['head_circumference', 'biparietal_diameter', 'femur_length', 'abdominal_circumference', 'occipitofrontal_diameter']
DEBUG:prenatalppkt.etl.extractors.observer:Starting Observer JSON extraction
DEBUG:prenatalppkt.etl.extractors.observer:Processing fetus 1
DEBUG:prenatalppkt.etl.extractors.observer:Found 6 measurements
DEBUG:prenatalppkt.etl.extractors.observer:Processing measurement: AC
DEBUG:prenatalppkt.etl.extractors.observer:AC has percentile=55.6% (valid)
DEBUG:prenatalppkt.etl.extractors.observer:Creating TermBin for AC: value=226.20000000000002mm, percentile=55.6%,


 STEP 1: Loading Observer JSON...
Loaded: tests/data/Apple_Sally_pretty.json
Fetuses: 1
Measurements: 6
Sample: AC =  22.62 cm

  STEP 2: Extracting biometry measurements to TermBins...
 Extracted 4 TermBins

  [1] AC: 226.2 mm (55.6%) at 26w6d [Fetus 1]
      HPO: HP:0034207 - Abnormal fetal gastrointestinal system morphology
      Normal: True

  [2] BPD: 66.8 mm (51.2%) at 26w6d [Fetus 1]
      HPO: HP:0000240 - Abnormality of skull size
      Normal: True

  [3] HC: 250.0 mm (42.5%) at 26w6d [Fetus 1]
      HPO: HP:0000240 - Abnormality of skull size
      Normal: True

  [4] Femur: 50.1 mm (46.8%) at 27w0d [Fetus 1]
      HPO: HP:0002823 - Abnormal femur morphology
      Normal: True

 STEP 3: Converting TermBins to PhenotypicFeatures...
 Generated 4 PhenotypicFeatures

  [1] HP:0034207
      Status: EXCLUDED (normal)
      Description: AC: 226.2 mm (55.6%) at 26w6d [Fetus 1]

  [2] HP:0000240
      Status: EXCLUDED (normal)
      Description: BPD: 66.8 mm (51.2%) at 26w6d [Fetus

# New

In [3]:
"""
PRENATALPPKT EXPANDED ETL PIPELINE
Observer JSON -> Biometry + Clinical Sections -> Phenopacket v2.0

Demonstrates the complete ETL pipeline:
1. Biometry extraction -> List[TermBin] -> quantitative HPO terms
2. Clinical indication -> reason for exam
3. Pregnancy dating -> LMP, EDD, gestational age context
4. Clinical impression -> qualitative HPO terms from free text
5. Phenopacket assembly -> GA4GH Phenopacket v2.0 JSON

Uses the official GA4GH phenopackets library per:
https://phenopacket-schema.readthedocs.io/en/latest/python.html
"""

import gzip
import json
import re
from datetime import datetime, timezone
from pathlib import Path

from google.protobuf.json_format import MessageToJson, Parse
from google.protobuf.timestamp_pb2 import Timestamp
import phenopackets.schema.v2 as pps2

# ETL Extractors (biometry -> TermBins)
from prenatalppkt.etl.extractors import observer

# ETL Section Parsers (clinical metadata -> Dicts)
from prenatalppkt.etl.sections import (
   parse_clinical_indication,
   parse_pregnancy_dating,
   parse_clinical_impression,
)

# HPO Concept Recognition
from prenatalppkt.hpo import HpoParser

# Gestational Age utilities
from prenatalppkt.gestational_age import GestationalAge

print("=" * 80)
print("PRENATALPPKT EXPANDED ETL PIPELINE")
print("Observer JSON -> Biometry + Clinical Sections -> Phenopacket v2.0")
print("=" * 80)

# =============================================================================
# STEP 1: Load HPO Concept Recognizer
# =============================================================================
print("\n[STEP 1] Loading HPO Concept Recognizer...")

HP_JSON_GZ = Path("tests/data/hp.json.gz")
TMP_HP_JSON = Path("/tmp/hp.json")

# Decompress hp.json.gz to temp location
with gzip.open(HP_JSON_GZ, "rt", encoding="utf-8") as f_in:
   with open(TMP_HP_JSON, "w", encoding="utf-8") as f_out:
       f_out.write(f_in.read())

hpo_parser = HpoParser(hpo_json_file=str(TMP_HP_JSON))
hpo_cr = hpo_parser.get_hpo_concept_recognizer()

print(f"  ? HPO version: {hpo_parser.get_version()}")
print(f"  ? Concept recognizer: {type(hpo_cr).__name__}")

# =============================================================================
# STEP 2: Load Observer JSON Data
# =============================================================================
print("\n[STEP 2] Loading Observer JSON...")

DATA_PATH = Path("tests/data/Apple_Sally_pretty.json")

with open(DATA_PATH) as f:
   observer_data = json.load(f)

# Keep raw JSON string for section parsers
with open(DATA_PATH) as f:
   observer_json_str = f.read()

print(f"  ? Loaded: {DATA_PATH}")
print(f"  ? Fetuses: {len(observer_data.get('fetuses', []))}")

first_fetus = observer_data["fetuses"][0]
measurements = first_fetus.get("measurements", [])
print(f"  ? Measurements: {len(measurements)}")
print(f"  ? Sample: {measurements[0]['label']} = {measurements[0]['value']} {measurements[0]['unit_of_measure']}")

# =============================================================================
# STEP 3: Extract Biometry -> TermBins
# =============================================================================
print("\n[STEP 3] Extracting biometry measurements to TermBins...")

term_bins = observer.extract(observer_data)

print(f"  ? Extracted {len(term_bins)} TermBins:")
for i, tb in enumerate(term_bins, 1):
   status = "? Normal" if tb.normal else "? Abnormal"
   print(f"    [{i}] {tb.hpo_id} ({tb.hpo_label}) - {status}")
   print(f"        {tb.description}")

# =============================================================================
# STEP 4: Parse Clinical Sections
# =============================================================================
print("\n[STEP 4] Parsing clinical sections...")

SOURCE_FORMAT = "observer_json"

# 4a: Clinical Indication
print("\n  --- Clinical Indication ---")
indication = parse_clinical_indication(observer_json_str, SOURCE_FORMAT)
indication_text = indication.get("indication_text", "")
if indication_text:
   print(f"  Indication: {indication_text[:100]}{'...' if len(indication_text) > 100 else ''}")
else:
   print("  Indication: (not found)")

# 4b: Pregnancy Dating
print("\n  --- Pregnancy Dating ---")
dating = parse_pregnancy_dating(observer_json_str, SOURCE_FORMAT)
print(f"  LMP: {dating.get('lmp', '(not found)')}")
print(f"  EDD: {dating.get('edd', '(not found)')}")
print(f"  Dating Method: {dating.get('dating_method', '(not found)')}")
print(f"  GA by Ultrasound: {dating.get('ga_by_ultrasound', '(not found)')}")

# 4c: Clinical Impression
print("\n  --- Clinical Impression ---")
impression = parse_clinical_impression(observer_json_str, SOURCE_FORMAT)
impression_text = impression.get("impression_text", "")

if impression_text:
   # Clean up for display
   preview = impression_text[:200].replace('\r', ' ').replace('\n', ' ')
   print(f"  Impression ({len(impression_text)} chars): \"{preview}...\"")
else:
   print("  Impression: (not found)")

print(f"  Growth Assessment: {impression.get('growth_assessment', '(not detected)')}")

# 4d: Extract HPO terms from clinical narrative
print("\n  --- HPO Concept Recognition from Clinical Text ---")
if impression_text:
   hpo_terms_from_text = hpo_cr.parse(impression_text)
   print(f"  Found {len(hpo_terms_from_text)} HPO terms in clinical narrative:")
   for term in hpo_terms_from_text:
       print(f"    ? {term.hpo_id}: {term.hpo_label}")
else:
   hpo_terms_from_text = []
   print("  (no impression text to parse)")

if not hpo_terms_from_text:
   print("  (no HPO terms matched)")

# =============================================================================
# STEP 5: Preview Anatomy Findings (Structured Data)
# =============================================================================
print("\n[STEP 5] Previewing anatomy findings...")

fetus_data = observer_data["fetuses"][0].get("fetus", {})
anatomy_list = fetus_data.get("anatomy", [])

normal_structures = []
abnormal_structures = []
unseen_structures = []
anomalies_found = []

for item in anatomy_list:
   main = item.get("main", {})
   label = main.get("label", "Unknown")
   state = main.get("anat_state", "")
   
   if state == "Normal":
       normal_structures.append(label)
   elif state == "Abnormal":
       abnormal_structures.append(label)
       # Check for specific anomalies
       anomalies = item.get("anomalies", [])
       if anomalies:
           for anom in anomalies:
               desc = anom.get("description", "?")
               anomalies_found.append(f"{label}: {desc}")
   elif state == "Unseen":
       unseen_structures.append(label)

print(f"  Normal ({len(normal_structures)}): {', '.join(normal_structures[:5])}...")
print(f"  Abnormal ({len(abnormal_structures)}): {', '.join(abnormal_structures) if abnormal_structures else '(none)'}")
print(f"  Not visualized ({len(unseen_structures)}): {', '.join(unseen_structures[:3])}...")

if anomalies_found:
   print(f"  ? Anomalies detected:")
   for anom in anomalies_found:
       print(f"    - {anom}")

print("  (Note: Anatomy section parser not yet implemented in ETL)")

# =============================================================================
# STEP 6: Convert to PhenotypicFeatures
# =============================================================================
print("\n[STEP 6] Converting to PhenotypicFeatures...")


def parse_ga_from_description(description: str, fallback_weeks: float = 26.9) -> tuple[int, int]:
   """Extract weeks and days from TermBin description."""
   match = re.search(r"at (\d+)w(\d+)d", description)
   if match:
       return int(match.group(1)), int(match.group(2))
   ga = GestationalAge.from_weeks(fallback_weeks)
   return ga.weeks, ga.days


# Get subject GA for features without specific timing
first_measurement = observer_data["fetuses"][0]["measurements"][0]
subject_ga_weeks = first_measurement.get("calculated_ega", 26.9)
subject_ga = GestationalAge.from_weeks(subject_ga_weeks)

phenotypic_features = []

# 6a: Convert biometry TermBins -> PhenotypicFeatures
print("\n  --- From Biometry ---")
for tb in term_bins:
   weeks, days = parse_ga_from_description(tb.description, subject_ga_weeks)
   
   pf = pps2.PhenotypicFeature(
       type=pps2.OntologyClass(id=tb.hpo_id, label=tb.hpo_label),
       excluded=tb.normal,  # normal=True means abnormality is EXCLUDED
       onset=pps2.TimeElement(
           gestational_age=pps2.GestationalAge(weeks=weeks, days=days)
       ),
       description=f"[Biometry] {tb.description}",
   )
   phenotypic_features.append(pf)

print(f"  ? Added {len(term_bins)} features from biometry")

# 6b: Convert clinical text HPO terms -> PhenotypicFeatures
print("\n  --- From Clinical Text ---")
text_feature_count = 0
for term in hpo_terms_from_text:
   # Findings mentioned in clinical impression are OBSERVED (not excluded)
   pf = pps2.PhenotypicFeature(
       type=pps2.OntologyClass(id=term.hpo_id, label=term.hpo_label),
       excluded=False,  # These are observed findings
       onset=pps2.TimeElement(
           gestational_age=pps2.GestationalAge(weeks=subject_ga.weeks, days=subject_ga.days)
       ),
       description=f"[Clinical Impression] Extracted from narrative text via HPO Concept Recognition",
   )
   phenotypic_features.append(pf)
   text_feature_count += 1

print(f"  ? Added {text_feature_count} features from clinical text")
print(f"\n  Total PhenotypicFeatures: {len(phenotypic_features)}")

# =============================================================================
# STEP 7: Build Complete Phenopacket v2.0
# =============================================================================
print("\n[STEP 7] Building Phenopacket v2.0...")

# Subject (fetus)
subject = pps2.Individual(
   id="fetus-1",
   sex=pps2.Sex.UNKNOWN_SEX,
   time_at_last_encounter=pps2.TimeElement(
       gestational_age=pps2.GestationalAge(weeks=subject_ga.weeks, days=subject_ga.days)
   ),
)

# Metadata
now = datetime.now(timezone.utc)
created_timestamp = Timestamp()
created_timestamp.FromDatetime(now)

hpo_resource = pps2.Resource(
   id="hp",
   name="Human Phenotype Ontology",
   url="http://purl.obolibrary.org/obo/hp.owl",
   version=hpo_parser.get_version() or "2025-01-01",
   namespace_prefix="HP",
   iri_prefix="http://purl.obolibrary.org/obo/HP_",
)

metadata = pps2.MetaData(
   created=created_timestamp,
   created_by="prenatalppkt-etl-pipeline",
   phenopacket_schema_version="2.0",
)
metadata.resources.append(hpo_resource)

# Assemble the Phenopacket
phenopacket = pps2.Phenopacket(
   id="apple-sally-fetus-1",
   subject=subject,
   meta_data=metadata,
)
phenopacket.phenotypic_features.extend(phenotypic_features)

print("  ? Phenopacket assembled successfully")
print(f"    ID: {phenopacket.id}")
print(f"    Subject: {phenopacket.subject.id} at {subject_ga.weeks}w{subject_ga.days}d")
print(f"    Features: {len(phenopacket.phenotypic_features)}")

# =============================================================================
# STEP 8: Output JSON
# =============================================================================
print("\n" + "=" * 80)
print("PHENOPACKET v2.0 OUTPUT (JSON)")
print("=" * 80)

phenopacket_json = MessageToJson(phenopacket, preserving_proto_field_name=True)
print(phenopacket_json)

# =============================================================================
# STEP 9: Validation & Summary
# =============================================================================
print("\n" + "=" * 80)
print("VALIDATION & SUMMARY")
print("=" * 80)

# Round-trip validation
print("\n[Validation] Round-trip test...")
parsed_back = Parse(phenopacket_json, pps2.Phenopacket())
assert parsed_back.id == phenopacket.id
assert len(parsed_back.phenotypic_features) == len(phenopacket.phenotypic_features)
print("  ? Round-trip validation passed")

# Feature breakdown
biometry_features = [pf for pf in phenopacket.phenotypic_features if "[Biometry]" in pf.description]
clinical_features = [pf for pf in phenopacket.phenotypic_features if "[Clinical" in pf.description]
excluded_count = sum(1 for pf in phenopacket.phenotypic_features if pf.excluded)
observed_count = len(phenopacket.phenotypic_features) - excluded_count

print("\n[Summary] Phenotypic Features:")
print(f"  Total: {len(phenopacket.phenotypic_features)}")
print(f"    From Biometry: {len(biometry_features)}")
print(f"    From Clinical Text: {len(clinical_features)}")
print(f"  Normal (excluded): {excluded_count}")
print(f"  Abnormal (observed): {observed_count}")

# Detailed feature list
print("\n[Detail] All Phenotypic Features:")
print("-" * 60)
for i, pf in enumerate(phenopacket.phenotypic_features, 1):
   status = "EXCLUDED (normal)" if pf.excluded else "OBSERVED (abnormal)"
   ga = pf.onset.gestational_age
   source = "Biometry" if "[Biometry]" in pf.description else "Clinical Text"
   print(f"\n  [{i}] {pf.type.id} - {pf.type.label}")
   print(f"      Source: {source}")
   print(f"      Status: {status}")
   print(f"      Onset: {ga.weeks}w{ga.days}d")

# Save to file
output_path = Path("output/apple_sally_phenopacket_expanded.json")
output_path.parent.mkdir(exist_ok=True)
with open(output_path, "w") as f:
   f.write(phenopacket_json)

print("\n" + "=" * 80)
print(f"SUCCESS: Phenopacket saved to {output_path}")
print("=" * 80)

PRENATALPPKT EXPANDED ETL PIPELINE
Observer JSON -> Biometry + Clinical Sections -> Phenopacket v2.0

[STEP 1] Loading HPO Concept Recognizer...


DEBUG:hpotk.util:Using default encoding 'utf-8'
DEBUG:hpotk.util:Opening /tmp/hp.json
DEBUG:hpotk.util:Looks like a local file: /tmp/hp.json
DEBUG:hpotk.util:Looks like decompressed data
DEBUG:hpotk.ontology.load.obographs._load:Extracting ontology terms
DEBUG:hpotk.ontology.load.obographs._factory:Unknown synonym type http://purl.obolibrary.org/obo/hp#allelic_requirement
DEBUG:hpotk.ontology.load.obographs._factory:Unknown synonym type http://purl.obolibrary.org/obo/hp#allelic_requirement
DEBUG:hpotk.ontology.load.obographs._factory:Unknown synonym type http://purl.obolibrary.org/obo/hp#allelic_requirement
DEBUG:hpotk.ontology.load.obographs._factory:Unknown synonym type http://purl.obolibrary.org/obo/hp#allelic_requirement
DEBUG:hpotk.ontology.load.obographs._factory:Unknown synonym type http://purl.obolibrary.org/obo/hp#allelic_requirement
DEBUG:hpotk.ontology.load.obographs._factory:Unknown synonym type http://purl.obolibrary.org/obo/hp#allelic_requirement
DEBUG:hpotk.ontology.io.o

  ? HPO version: 2025-10-22
  ? Concept recognizer: HpoExactConceptRecognizer

[STEP 2] Loading Observer JSON...
  ? Loaded: tests/data/Apple_Sally_pretty.json
  ? Fetuses: 1
  ? Measurements: 6
  ? Sample: AC = 22.62 cm

[STEP 3] Extracting biometry measurements to TermBins...
  ? Extracted 4 TermBins:
    [1] HP:0034207 (Abnormal fetal gastrointestinal system morphology) - ? Normal
        AC: 226.2 mm (55.6%) at 26w6d [Fetus 1]
    [2] HP:0000240 (Abnormality of skull size) - ? Normal
        BPD: 66.8 mm (51.2%) at 26w6d [Fetus 1]
    [3] HP:0000240 (Abnormality of skull size) - ? Normal
        HC: 250.0 mm (42.5%) at 26w6d [Fetus 1]
    [4] HP:0002823 (Abnormal femur morphology) - ? Normal
        Femur: 50.1 mm (46.8%) at 27w0d [Fetus 1]

[STEP 4] Parsing clinical sections...

  --- Clinical Indication ---
  Indication: (not found)

  --- Pregnancy Dating ---
  LMP: 0001-01-01
  EDD: None
  Dating Method: None
  GA by Ultrasound: None

  --- Clinical Impression ---
  Impression 